# Ruby on Rails Code Agents Proof of Concept

Generated code is better if the you choose a more powerful model.
To reduce the cost this demo is using `gpt-4.1-nano`.

TODO: Save the generated files



In [ ]:
# --- Imports ---
import gradio as gr
import os
import openai
from dotenv import load_dotenv
from pathlib import Path
from rails_agents import RailsAgents

# --- Initialize environment ---
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

# --- Configuration ---
client = openai.OpenAI()
MODEL = "gpt-4.1-nano"
PROJECT_ROOT = "rails_workspace"



In [ ]:
# --- File management utilities ---
if not os.path.exists(PROJECT_ROOT):
    os.makedirs(PROJECT_ROOT)

def write_to_file(file_path, content):
    """Utility to write code to the local filesystem."""
    full_path = Path(PROJECT_ROOT) / file_path
    full_path.parent.mkdir(parents=True, exist_ok=True)
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(content)
    return f"Successfully wrote to {file_path}"

def list_files():
    """Helper to show the current project structure in the IDE."""
    file_list = []
    for root, dirs, files in os.walk(PROJECT_ROOT):
        for file in files:
            file_list.append(os.path.join(root, file).replace(PROJECT_ROOT + "/", ""))
    return "\n".join(file_list) if file_list else "No files generated yet."


In [ ]:
# --- Rails Helpers Coders Logic ---
def process_rails_request(user_input):
    logs = []
    
    # 1. Planning
    logs.append("--- [Planning Agent] Starting Orchestration ---")
    plan = RailsAgents.planning_agent(user_input)
    logs.append(plan)
    
    # 2. Backend Generation
    logs.append("\n--- [Backend Engineer] Generating REST API ---")
    be_code = RailsAgents.backend_agent(plan)
    logs.append(be_code)
    
    # 3. Frontend Generation
    logs.append("\n--- [Frontend Engineer] Generating UI ---")
    fe_code = RailsAgents.frontend_agent(plan)
    logs.append(fe_code)
    
    # 4. QA Generation
    logs.append("\n--- [QA Tester] Generating RSpec Tests ---")
    qa_code = RailsAgents.qa_agent(plan)
    logs.append(qa_code)
    
    return "\n".join(logs)


In [ ]:
# --- Gradio UI ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛤️ Ruby on Rails AI Agents")
    gr.Markdown("Orchestrate LLM agents to build your Rails application locally.")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Project Explorer")
            file_explorer = gr.Textbox(label="Local Files", value=list_files(), lines=20, interactive=False)
            refresh_btn = gr.Button("Refresh File Tree")
            
        with gr.Column(scale=3):
            user_input = gr.Textbox(
                label="What are we building today?", 
                placeholder="e.g., Create a blog system with Posts and Comments including a REST API...",
                lines=3
            )
            run_btn = gr.Button("Generate Project", variant="primary")
            
            output_console = gr.Code(
                label="Agent Console Logs", 
                language="markdown",
                interactive=False,
                lines=25
            )

    # Event Handlers
    run_btn.click(
        fn=process_rails_request,
        inputs=user_input,
        outputs=output_console
    )
    
    refresh_btn.click(
        fn=list_files,
        outputs=file_explorer
    )

if __name__ == "__main__":
    demo.launch(inbrowser=True)
